In [1]:
import actionet
import scanpy as sc
import anndata
import pandas as pd
import numpy as np

import lets_plot
lets_plot.LetsPlot.setup_html(no_js=True)

In [ ]:
adata_src = anndata.read_h5ad("../data/adata_agg_Scn4b_OX_fil.h5ad", backed = "r")
adata_src.copy(filename="../data/adata_working_test_lazy.h5ad")
adata_src.file.close()

In [3]:
adata = anndata.read_h5ad("../data/adata_working_test_lazy.h5ad", backed = "r+")

In [4]:
actionet.filter_anndata(adata, min_cells_per_feat=0.01, inplace=True)

In [5]:
lzt = actionet.create_lazy_transform(adata, target_sum=1e4, log_base=2, pseudocount=1)

In [ ]:
actionet.reduce_kernel(adata, n_components=30, key_added='action', inplace=True, backed_n_threads=8, svd_algorithm="halko", max_iter=0, lazy_transform=lzt)
adata

Computing reduced ACTION kernel (operator):
Halko (operator) -- A: 6790 x 14409
	Iteration 20/20
Computing reduced ACTION kernel from precomputed SVD (operator):
Kernel computed successfully.


AnnData object with n_obs × n_vars = 6790 × 14409 backed at '../data/adata_working_test_lazy.h5ad'
    obs: 'Barcode', 'CellLabel'
    var: 'ENSEMBL', 'Gene', 'Chromosome', 'Biotype'
    uns: 'action_params'
    obsm: 'action', 'action_B'
    varm: 'action_U', 'action_A'

In [ ]:
actionet.correct_batch_effect(adata, batch_key="UID", inplace=True, lazy_transform=lzt)

In [ ]:
actionet.run_actionet(adata, compute_specificity_parallel=True, inplace=True, layout_epochs=200, reduction_key="action_corrected", lazy_transform=lzt)

Running ACTION decomposition...
Running ACTION (14 threads):
	Iterating from k = 2 ... 30
	Completed: 29/29 (100.0%)  
Joining trace of C & H matrices (depth = 30) ... done (464 archetypes)
Pruning archetypes:
	Non-specific archetypes: 4
	Unreproducible archetypes: 59
	Trivial archetypes: 0
Merging 401 archetypes:
Archetypes in merged set: 24
Building network...
Building adaptive network (density = 1.00)
	Building index ... done
	Constructing adaptive-nearest neighbor graph ... done
	Finalizing network ... done
Computing archetype footprints via diffusion...
Computing 2D UMAP layout...
Optimizing layout using method 'umap': 2 components 
UMAP embedding parameters a = 0.115, b = 1.929, gamma = 1.000
Optimizing for 100 epochs with 14 threads 
Optimizing with Adam:
	 alpha = 1.000,  beta1 = 0.500, beta2 = 0.900, eps = 1.000e-07
Optimization finished
Computing 3D UMAP layout...
Optimizing layout using method 'umap': 3 components 
UMAP embedding parameters a = 0.115, b = 1.929, gamma = 1.00

In [9]:
actionet.plot_umap(adata, color='CellLabel', basis='umap_2d_actionet')


In [10]:
markers = actionet.find_markers(adata, adata.obs['CellLabel'], features_use="Gene", top_genes=30, return_type='dataframe')
markers

Computing feature specificity (backed sparse) ... done


,CT_1,CT_10,CT_11,CT_12,CT_13,CT_14,CT_15,CT_16,CT_2,CT_3,CT_4,CT_5,CT_6,CT_7,CT_8,CT_9
0,Slc1a2,Adarb2,Plp1,Xylt1,Sox2ot,Lingo2,Penk,Mrc1,Bsg,Adamts20,Slc38a2,Cpne4,Grip1,Sox2ot,Plxdc2,Dlc1
1,Msi2,Pcdh15,Mbp,Lhfpl3,Adarb2,Ebf1,Drd2,Slc9a9,Igf1r,Cfap299,Cemip,Clstn2,Erbb4,Sox6,Hexb,Atp13a5
2,Gpc5,Olfm3,Plcl1,Tnr,2610307P16Rik,Sntg2,Grik3,Stab1,Tmsb4x,Spag16,Ptgds,Asic2,Il1rapl2,Col25a1,Tmcc3,Cald1
3,Apoe,Erbb4,Edil3,Dscam,Pbx1,Nrg3,Adora2a,P2rx7,mt-Co1,Cfap54,Fbxl7,Inpp4b,Astn2,Nos1,Tgfbr1,Pdgfrb
4,Slc1a3,Trpm3,Fnbp1,Ptprz1,Sdk2,Reln,Chrm3,Rbpj,Sgms1,Ccdc162,Slc4a10,Kcnt2,Tcf4,Ptprt,Inpp5d,Colec12
5,Plpp3,Ntng1,Tmeff2,Epn2,Nfib,Cntnap3,Nell1,Dab2,Ptprg,Spef2,Cped1,Prkca,Pcdh15,Aff2,Lrmda,Dock6
6,Prex2,Foxp2,Mast4,Vcan,Sox1ot,Onecut2,Sgcz,Gab2,mt-Co2,Cfap61,Eya2,Ptchd4,Dlgap1,Zfp804b,Cx3cr1,Phldb2
7,Nfia,Tshz1,Rnf220,Arhgap31,Ppm1e,Samd5,Necab1,Anapc15,Spock2,Kif6,Slc7a11,Galntl6,Nxph1,Tox,Apbb1ip,Atp1a2
8,Mir99ahg,Alk,St6galnac3,Tmem132d,Dlx6os1,Dgkz,Cacnb2,Ctsc,mt-Co3,Cfap46,Trpm3,Slc7a14,Ppm1l,Sst,Lyn,Plxdc2
9,Cpe,Sgcd,Pex5l,Maml2,Tcf4,Ccdc88c,Ptprm,Tnfrsf11a,Gpcpd1,Ak7,Sned1,Fam155a,Srrm4,Tenm1,Rreb1,Vtn


In [11]:
annots_out = actionet.annotate_cells(adata, markers, layer = None, method='vision', features_use='Gene', use_enrichment=True, use_lpa=False, n_threads=0)

In [12]:
actionet.plot_umap(adata, color=annots_out['labels'], basis='umap_2d_actionet')

In [ ]:
actionet.compute_feature_specificity(adata, labels = "full_label", n_threads=1, return_raw=True, lazy_transform=lzt)

In [ ]:
actionet.compute_feature_specificity(adata, labels = "full_label", n_threads=8, return_raw=True, lazy_transform=lzt)

In [ ]:
actionet.compute_archetype_feature_specificity()